In [1]:
import sys
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

sys.path.append(os.path.abspath('..')) 

try:
    from src.trainer import get_all_pipelines
    print("Function 'get_all_pipelines' imported successfully!")
except ImportError as e:
    print(f" Failed to import: {e}. Check if trainer.py has the function!")


df = pd.read_csv('../data/processed_sarcasm.csv').dropna()

# Using 'text' and 'label' 
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)


pipelines = get_all_pipelines()
best_f1 = 0
best_model = None
best_name = ""
results = []

print("Benchmarking Models...")
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    results.append({"Model": name, "Accuracy": acc, "F1-Score": f1})
    print(f"{name}: Accuracy={acc:.4f}, F1={f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model = pipe
        best_name = name


os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/sarcasm_model.pkl')

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)


results_df.to_csv('../models/model_comparison.csv', index=False)

print("\n MODEL COMPARISON SUMMARY:")
print("-" * 45)
print(results_df.to_string(index=False))
print("-" * 45)

with open("../models/winner_name.txt", "w") as f:
    f.write(best_name)

print(f"\nThe best model is {best_name}. Saved!")

Function 'get_all_pipelines' imported successfully!
Benchmarking Models...


Logistic Regression: Accuracy=0.7084, F1=0.6926


SVM (LinearSVC): Accuracy=0.6986, F1=0.6891


Decision Tree: Accuracy=0.6226, F1=0.5547


K-Means Clustering: Accuracy=0.4601, F1=0.2784



 MODEL COMPARISON SUMMARY:
---------------------------------------------
              Model  Accuracy  F1-Score
Logistic Regression  0.708368  0.692556
    SVM (LinearSVC)  0.698553  0.689061
      Decision Tree  0.622650  0.554712
 K-Means Clustering  0.460136  0.278388
---------------------------------------------

The best model is Logistic Regression. Saved!
